In [1]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import random
import shutil
import hashlib
from collections import defaultdict
import json
import warnings
warnings.filterwarnings('ignore')

print(" Ready")

 Ready


In [2]:
VIDEO_PATH = Path("./dataset")
OUTPUT_PATH = Path("./herbal_domain_dataset")

if OUTPUT_PATH.exists():
    shutil.rmtree(OUTPUT_PATH)

(source_path := OUTPUT_PATH / "source" / "images").mkdir(parents=True)
(target_path := OUTPUT_PATH / "target" / "images").mkdir(parents=True)
(test_path := OUTPUT_PATH / "test" / "images").mkdir(parents=True)
(OUTPUT_PATH / "source").mkdir(exist_ok=True)
(OUTPUT_PATH / "test").mkdir(exist_ok=True)

print(f" Output: {OUTPUT_PATH}")

 Output: herbal_domain_dataset


In [3]:
VIDEO_TO_SPECIES = {
    1: {"name": "Salvia", "file": "VID_20260408_171126.mp4"},
    2: {"name": "Gleditsia_triacanthos", "file": "VID_20260408_171454.mp4"},
    3: {"name": "Lophostemon_confertus", "file": "VID_20260408_171606.mp4"},
    4: {"name": "Physalis_peruviana", "file": "VID_20260408_171903.mp4"},
    5: {"name": "Little_Ironweed", "file": "VID_20260408_172029.mp4"},
    6: {"name": "Leucaena", "file": "VID_20260408_172233.mp4"},
    7: {"name": "White_Goosefoot", "file": "VID_20260408_172556.mp4"},
    8: {"name": "Ashwagandha", "file": "VID_20260408_172740.mp4"},
    9: {"name": "Mixed_Asparagus_Mexican_Tea", "file": "VID_20260408_172835.mp4"},
    10: {"name": "Mint", "file": "VID_20260408_172918.mp4"},
    11: {"name": "Ajwain", "file": "VID_20260408_173112.mp4"},
    12: {"name": "Ashwagandha", "file": "VID_20260408_173310.mp4"},
    13: {"name": "Mint", "file": "VID_20260408_173405.mp4"},
    14: {"name": "Ajwain", "file": "VID_20260408_173452.mp4"},
    15: {"name": "Tulsi", "file": "VID_20260408_173854.mp4"},
    16: {"name": "Giloy", "file": "VID_20260408_173936.mp4"},
    17: {"name": "Cardiospermum_halicacabum", "file": "VID_20260408_174106.mp4"},
    18: {"name": "Kalmegh", "file": "VID_20260408_174147.mp4"},
    19: {"name": "Bathua", "file": "VID_20260408_174235.mp4"},
    20: {"name": "Insulin_Plant", "file": "VID_20260408_174323.mp4"},
    21: {"name": "Neem", "file": "VID_20260408_174524.mp4"},
}

# Create species ID mapping
unique_species = sorted(set(v["name"] for v in VIDEO_TO_SPECIES.values()))
species_to_idx = {s: i for i, s in enumerate(unique_species)}
idx_to_species = {i: s for s, i in species_to_idx.items()}

print(f" {len(VIDEO_TO_SPECIES)} videos, {len(unique_species)} species")

 21 videos, 18 species


In [4]:
def get_video_hash(path, size=1024*1024):
    with open(path, 'rb') as f:
        f.seek(0)
        head = f.read(size)
        f.seek(-min(size, path.stat().st_size), 2)
        tail = f.read(size)
        return hashlib.md5(head + tail).hexdigest()

def get_unique_videos(folder):
    videos = []
    for ext in ['*.mp4', '*.avi', '*.mov', '*.MP4']:
        videos.extend(folder.glob(ext))
    
    unique, seen = [], set()
    for v in videos:
        h = get_video_hash(v)
        if h not in seen:
            seen.add(h)
            unique.append(v)
    return unique

all_videos = get_unique_videos(VIDEO_PATH)
print(f"Found {len(all_videos)} unique videos")

# Map video files to IDs
video_map = {}
for v in all_videos:
    for vid, info in VIDEO_TO_SPECIES.items():
        if info["file"] == v.name:
            video_map[vid] = v
            break

valid_ids = sorted(video_map.keys())
print(f" Matched {len(valid_ids)} videos")

Found 21 unique videos
 Matched 21 videos


In [5]:
def quality_score(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    score = 0
    brightness = np.mean(gray)
    contrast = np.std(gray)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    saturation = np.std(hsv[:, :, 1])
    
    if 80 <= brightness <= 170: score += 2
    elif 60 <= brightness <= 200: score += 1
    
    if contrast >= 40: score += 2
    elif contrast >= 25: score += 1
    
    if sharpness >= 80: score += 2
    elif sharpness >= 50: score += 1
    
    if saturation >= 35: score += 1
    
    return score

In [6]:
print("Extracting frames...")
all_frames = []

for vid in valid_ids:
    video_path = video_map[vid]
    species = VIDEO_TO_SPECIES[vid]["name"]
    cap = cv2.VideoCapture(str(video_path))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    interval = max(1, fps // 2)
    extracted = 0
    
    for frame_num in range(0, total, interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        if not ret or extracted >= 100:
            break
        
        score = quality_score(frame)
        all_frames.append({
            'video_id': vid,
            'species': species,
            'frame': frame,
            'quality': score,
            'filename': f"vid{vid:03d}_{species}_frame{frame_num:06d}.jpg"
        })
        extracted += 1
    
    cap.release()
    print(f"  Video {vid}: {extracted} frames")

print(f"\nTotal frames: {len(all_frames)}")

# Split by quality
scores = [f['quality'] for f in all_frames]
source_thresh = np.percentile(scores, 65)
target_thresh = np.percentile(scores, 35)

source_frames = [f for f in all_frames if f['quality'] >= source_thresh]
target_frames = [f for f in all_frames if f['quality'] <= target_thresh]
medium_frames = [f for f in all_frames if source_thresh > f['quality'] > target_thresh]

print(f"Source: {len(source_frames)} | Target: {len(target_frames)} | Skipped: {len(medium_frames)}")

Extracting frames...
  Video 1: 36 frames
  Video 2: 37 frames
  Video 3: 40 frames
  Video 4: 40 frames
  Video 5: 33 frames
  Video 6: 54 frames
  Video 7: 41 frames
  Video 8: 25 frames
  Video 9: 37 frames
  Video 10: 57 frames
  Video 11: 42 frames
  Video 12: 68 frames
  Video 13: 75 frames
  Video 14: 58 frames
  Video 15: 39 frames
  Video 16: 66 frames
  Video 17: 55 frames
  Video 18: 28 frames
  Video 19: 53 frames
  Video 20: 69 frames
  Video 21: 85 frames

Total frames: 1038
Source: 514 | Target: 524 | Skipped: 0


In [7]:
print("Saving source images")
labels = []

for f in source_frames:
    cv2.imwrite(str(source_path / f['filename']), f['frame'])
    labels.append({
        'filename': f['filename'],
        'species': f['species'],
        'species_id': species_to_idx[f['species']],
        'video_id': f['video_id']
    })

df = pd.DataFrame(labels)
df.to_csv(OUTPUT_PATH / "source" / "labels.csv", index=False)
df.to_csv(OUTPUT_PATH / "source" / "labels.txt", sep='\t', index=False)

print(f" Saved {len(labels)} labeled source images")
print("\nLabel distribution:")
print(df['species'].value_counts().to_string())

Saving source images
 Saved 514 labeled source images

Label distribution:
species
Mint                           71
Neem                           55
Leucaena                       49
Ashwagandha                    47
Physalis_peruviana             39
Lophostemon_confertus          38
Gleditsia_triacanthos          36
Little_Ironweed                33
Mixed_Asparagus_Mexican_Tea    33
Ajwain                         25
Giloy                          25
White_Goosefoot                20
Bathua                         19
Kalmegh                         9
Cardiospermum_halicacabum       8
Salvia                          4
Insulin_Plant                   2
Tulsi                           1


In [8]:
print("Saving target images")
for f in target_frames:
    cv2.imwrite(str(target_path / f['filename']), f['frame'])

print(f"Saved {len(target_frames)} unlabeled target images")

Saving target images
Saved 524 unlabeled target images


In [9]:
test_size = max(10, int(0.2 * len(target_frames)))
test_frames = random.sample(target_frames, test_size)

test_labels = []
for i, f in enumerate(test_frames):
    filename = f"test_{i:04d}_{f['filename']}"
    cv2.imwrite(str(test_path / filename), f['frame'])
    test_labels.append({
        'filename': filename,
        'species': f['species'],
        'species_id': species_to_idx[f['species']],
        'video_id': f['video_id']
    })

pd.DataFrame(test_labels).to_csv(OUTPUT_PATH / "test" / "test_labels.csv", index=False)
print(f" Saved {len(test_frames)} test images with labels")

 Saved 104 test images with labels


In [10]:
print("DATASET READY")
print(f"Location: {OUTPUT_PATH}")
print(f"\nSource (labeled): {len(source_frames)} images")
print(f"Target (unlabeled): {len(target_frames)} images")
print(f"Test: {len(test_frames)} images")
print(f"\nSpecies: {len(unique_species)}")
for s, idx in species_to_idx.items():
    cnt = len([l for l in labels if l['species'] == s])
    print(f"  {idx}: {s[:25]:25s} ({cnt} source)")

# Save metadata
with open(OUTPUT_PATH / "metadata.json", 'w') as f:
    json.dump({
        'species_map': species_to_idx,
        'source_count': len(source_frames),
        'target_count': len(target_frames),
        'test_count': len(test_frames)
    }, f, indent=2)

print("\n Complete! Use source/ for training, target/ for adaptation, test/ for evaluation")

DATASET READY
Location: herbal_domain_dataset

Source (labeled): 514 images
Target (unlabeled): 524 images
Test: 104 images

Species: 18
  0: Ajwain                    (25 source)
  1: Ashwagandha               (47 source)
  2: Bathua                    (19 source)
  3: Cardiospermum_halicacabum (8 source)
  4: Giloy                     (25 source)
  5: Gleditsia_triacanthos     (36 source)
  6: Insulin_Plant             (2 source)
  7: Kalmegh                   (9 source)
  8: Leucaena                  (49 source)
  9: Little_Ironweed           (33 source)
  10: Lophostemon_confertus     (38 source)
  11: Mint                      (71 source)
  12: Mixed_Asparagus_Mexican_T (33 source)
  13: Neem                      (55 source)
  14: Physalis_peruviana        (39 source)
  15: Salvia                    (4 source)
  16: Tulsi                     (1 source)
  17: White_Goosefoot           (20 source)

 Complete! Use source/ for training, target/ for adaptation, test/ for evaluation
